# Pipeline complet NdapuSpeech — exécution bout en bout

Ce notebook exécute **toutes les étapes** du pipeline ASR Wolof dans l'ordre :

| # | Étape | Durée estimée |
|---|-------|---------------|
| 1 | Téléchargement des datasets existants | ~10 min |
| 2 | Collecte YouTube (optionnel) | ~30 min |
| 3 | Construction du dataset unifié | ~5 min |
| 4 | Entraînement tokenizer BPE Wolof | ~2 min |
| 5 | Fine-tuning Whisper sur le Wolof | ~heures (GPU requis) |
| 6 | Entraînement KenLM n-gram | ~5 min |
| 7 | Évaluation WER/CER vs M-Kiriku | ~5 min |
| 8 | Inférence sur des fichiers audio | ~1 min |

> **Prérequis** : Python ≥ 3.11, GPU CUDA disponible (RTX recommandé),
> `ffmpeg` et `yt-dlp` installés dans le PATH.

## Prérequis — installation des packages

In [1]:
%pip install -e "C:/Users/ROG ZEPHYRUS G14/OneDrive - Université Cheikh Anta DIOP de DAKAR/Desktop/NdapuSpeech"[audio]

Obtaining file:///C:/Users/ROG%20ZEPHYRUS%20G14/OneDrive%20-%20Universit%C3%A9%20Cheikh%20Anta%20DIOP%20de%20DAKAR/Desktop/NdapuSpeech
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ndapuspeech (pyproject.toml): started
  Building editable for ndapuspeech (pyproject.toml): finished with status 'done'
  Created wheel for ndapuspeech: filename=ndapuspeech-0.1.0-0.editable-py3-none-any.whl size=3571 sha256=01b868f8eb5d53f4330ebe6961231999e26d05146175a235d2c32707a49020ab
  Stored in directory: C:\Users\ROG ZEPHYRUS G14\AppDat

In [ ]:
import logging, time, torch

logging.basicConfig(level=logging.INFO, format='%(message)s')
logging.getLogger().setLevel(logging.INFO)

from ndapuspeech import config, data, models, evaluation, inference
from ndapuspeech.training import TrainingConfig, train_model
from ndapuspeech.kenlm import train_kenlm

config.ensure_dirs()

print(f"CUDA : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"Base model : {config.BASE_MODEL}")
print(f"Language : {config.LANGUAGE}")
print(f"Config chargée : {len(config.get_config())} sections")

---
## Étape 1 — Téléchargement des datasets

In [ ]:
t0 = time.time()
results = data.download_all()
print(f"\n--- Téléchargements ({time.time()-t0:.0f}s) ---")
for source, path in results.items():
    status = 'OK' if path else 'SKIPPED'
    print(f"  {source:16s} {status}")

---
## Étape 3 — Construction du dataset unifié train / val / test

In [ ]:
t0 = time.time()
summary = data.build_unified_dataset(
    output_dir=config.PROCESSED_DATA_DIR / 'ndapuspeech_ds',
    train_ratio=config.TRAIN_SPLIT,
)
print(f"\nDataset ({time.time()-t0:.0f}s) :")
print(f"  Segments  : {summary['n_samples']}")
print(f"  Durée tot : {summary['total_hours']:.1f} h")
for src, hrs in sorted(summary['by_source'].items(), key=lambda x: -x[1]):
    print(f"  {src:20s} {hrs:6.1f} h")
print(f"  Splits    : {summary['splits']}")

---
## Étape 4 — Entraînement du tokenizer BPE Wolof

In [ ]:
t0 = time.time()
tok_result = models.train_tokenizer_pipeline(
    vocab_size=config.DATA['tokenizer']['vocab_size'],
    extend_whisper=True,
)
print(f"\nTokenizer ({time.time()-t0:.0f}s) :")
print(f"  Pièces BPE   : {tok_result['n_pieces']}")
print(f"  Corpus chars  : {tok_result['corpus_chars']}")
print(f"  Étendu vers   : {tok_result['extended_tokenizer']}")

---
## Étape 5 — Fine-tuning Whisper sur le Wolof

Pour un vrai entraînement, supprimez les lignes de `num_epochs`/`max_steps`
et laissez les valeurs par défaut dans `configs/training.yaml`.

In [ ]:
cfg = TrainingConfig.from_yaml({
    'num_epochs': 20,
    'eval_steps': 200,
    'save_steps': 400,
})
print(cfg)

In [ ]:
t0 = time.time()
out_dir = train_model(cfg)
print(f"\nModèle sauvegardé : {out_dir} ({(time.time()-t0)/60:.0f} min)")

---
## Étape 6 — Entraînement KenLM n-gram

In [ ]:
arpa_path, bin_path = train_kenlm()
print(f"ARPA : {arpa_path}")
print(f"BIN  : {bin_path}")

---
## Étape 7 — Évaluation WER / CER vs M-Kiriku

In [ ]:
from ndapuspeech.data import load_jsonl

test_samples = load_jsonl(config.PROCESSED_DATA_DIR / 'ndapuspeech_ds' / 'test.jsonl')
print(f'{len(test_samples)} échantillons de test')

summaries = evaluation.evaluate_and_compare(
    models=[
        str(out_dir),
        'AIHubSN/M-Kiriku-ASR',
    ],
    samples=test_samples,
    batch_size=8,
)

In [ ]:
import pandas as pd

rows = [
    {'modèle': m, 'WER': s['overall']['wer'], 'CER': s['overall']['cer']}
    for m, s in summaries.items()
]
df = pd.DataFrame(rows)
display(df)

baseline_wer = summaries.get('AIHubSN/M-Kiriku-ASR', {}).get('overall', {}).get('wer', None)
ours_wer = summaries.get(str(out_dir), {}).get('overall', {}).get('wer', None)
if baseline_wer and ours_wer:
    verdict = 'BEATS baseline' if ours_wer < baseline_wer else 'sous la baseline'
    print(f"\nVerdict : {verdict}")
    print(f"  M-Kiriku : {baseline_wer:.2f}%")
    print(f"  NdapuSpeech : {ours_wer:.2f}%")

---
## Étape 8 — Inférence sur des fichiers audio

In [ ]:
import glob

audio_files = sorted(glob.glob('data/youtube/segments/*.wav'))[:5]
if not audio_files:
    print('Aucun fichier audio trouvé dans data/youtube/segments/')
else:
    results = inference.transcribe_files(
        model_path=str(out_dir),
        audio_paths=audio_files,
        lm_path=str(bin_path) if bin_path else None,
    )
    for path, r in results.items():
        print(f'\n--- {path} ---')
        print(r['text'])

---
## Résumé

Le pipeline complet est terminé. Pour un entraînement sérieux :

1. Mettez `num_epochs=20` (ou plus) dans `configs/training.yaml`
2. Activez la collecte YouTube (décommentez l'étape 2)
3. Relancez ce notebook sur un GPU puissant (≥ 16 Go VRAM)
4. Poussez le modèle sur HuggingFace Hub

```python
# Push vers Hub (décommenter)
# from huggingface_hub import login; login()
# model.push_to_hub('ndapuspeech/ndapuspeech-asr')
# processor.push_to_hub('ndapuspeech/ndapuspeech-asr-v1')
```